<h4>Import the necessary libraries</h4>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV,StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

from iclltuning import iclltuning

<h4>Read the file</h4>

In [2]:
train_df=pd.read_csv(r"D:\Code_files\Bank_Term_Deposit\train.csv",low_memory=False,delimiter=',')
#train_df.head().to_string()
#X_train=train_df[['age','balance','duration']].values.copy()

train_df['y']=np.where(train_df['y']=='no',0,1)

In [3]:
train_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0


In [4]:
def convertData(func_df):
    func_df['marital']=np.where(func_df['marital']=='married',1,0)
    func_df['default']=np.where(func_df['default']=='yes',1,0)
    func_df['housing']=np.where(func_df['housing']=='yes',1,0)
    func_df['loan']=np.where(func_df['loan']=='yes',1,0)
    func_df['poutcome']=np.where(func_df['poutcome']=='success',1,0)

    return func_df

In [5]:
selected_columns=[
    'age','marital','default',
    'housing','loan','day',
    'campaign','pdays','previous','poutcome'
]

X_train=train_df[selected_columns].copy()
y_train=train_df['y'].copy()

In [6]:
X_train=convertData(X_train)

<h4>See class distribution</h4>

In [7]:
(train_df['y'].value_counts(normalize=True)*100).round(1).astype(str)+'%'

y
0    88.3%
1    11.7%
Name: proportion, dtype: object

<h4>Define the ML Models</h4>

In [8]:
rf_model=RandomForestClassifier(class_weight='balanced')
lr_model=LogisticRegression(penalty='elasticnet',l1_ratio=0.5,class_weight='balanced',random_state=0,solver='saga',max_iter=10000)

<h4>Initialize Grid Search CV object</h4>

In [9]:
# Define the Hyperparameter Grid for the Random Forest classifier
rf_param_grid={
    'n_estimators' : [10,50,100],
    'max_depth':[3,5,7],
    'max_features' : ['sqrt',None]
}

# Define K-Fold object
skf_obj=StratifiedKFold(n_splits=5,shuffle=True,random_state=0)

# Initialize Grid Search CV object
cv_obj=GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    scoring='recall',
    refit=True,
    n_jobs=-1,
    cv=skf_obj
)

<h4>Define SMOTE object for the second layer</h4>

In [10]:
sm_obj = SMOTE(random_state=0)

<h4>Define Sample Weights</h4>

In [11]:
# Set weights
smpl_weights=y_train.copy()

# Optimize the model for the situation where the population psoitive class ratio is actually 5%.
smpl_weights=np.array(np.where(smpl_weights==1,0.05/0.11,(1-0.05)/(1-0.11)))

<h4>Define ICLL object and fit it to the training data</h4>

In [12]:
icll_model=iclltuning(
    model_l1=None, # No model is requeired in Layer 1 as it is already present in the GridSearchCV object
    model_l2=lr_model, # Logistic Regression Model for Layer 2
    cv_obj_l1=cv_obj, # GridSearchCV object to perform hyperparameter tuning on Random Forest model in Layer 1
    cv_obj_l2=None, # No hyperparameter tuning required for Layer 2
    smote_obj_l1=None, # No Resampling for Layer 1
    smote_obj_l2=sm_obj, # Resampling to be applied to Layer 2
    sample_wt_l1=smpl_weights, # Assign sample weights in training Layer 1 model
    sample_wt_l2=None, # No sample weights to be assigned for training Layer 2 model
    custom_distance_threshold=None # Let ICLL model compute the optimal threshold to construct flat clusters
)

icll_model.fit(X_train,y_train)

<h4>Make Predictions</h4>

In [13]:
# Read the test data
test_df=pd.read_csv(r"D:\Code_files\Bank_Term_Deposit\test.csv",low_memory=False,delimiter=',')

# Convert the target label to 1/0 format
test_df['y']=np.where(test_df['y']=='no',0,1)

# Transform the data
X_test=test_df[selected_columns].copy()
y_test=test_df['y'].copy()

X_test=convertData(X_test)

<h4>Use .predict() to get hard classification labels</h4>

In [14]:
y_hard_labels=icll_model.predict(X_test)
y_soft_labels=icll_model.predict_proba(X_test)

In [15]:
y_soft_labels

array([0.33580338, 0.05960329, 0.0946222 , ..., 0.04190832, 0.18995429,
       0.09210212])

<h4>Showcase various methods of iclltuning object</h4>

In [16]:
fitted_cv_obj=icll_model.get_fitted_cv_layer_object('L1')

cv_fit_results=pd.DataFrame(fitted_cv_obj.cv_results_)

#cv_fit_results.head().to_string()

cv_fit_results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.243994,0.057331,0.016515,0.004726,3,sqrt,10,"{'max_depth': 3, 'max_features': 'sqrt', 'n_es...",0.831646,0.848466,0.838187,0.826195,0.838629,0.836625,0.007486,12
1,0.648893,0.135473,0.021018,0.002614,3,sqrt,50,"{'max_depth': 3, 'max_features': 'sqrt', 'n_es...",0.828687,0.836163,0.839433,0.830089,0.838474,0.834569,0.004384,13
2,1.163523,0.153313,0.038560,0.007382,3,sqrt,100,"{'max_depth': 3, 'max_features': 'sqrt', 'n_es...",0.827441,0.837876,0.834917,0.827285,0.834268,0.832357,0.004255,15
3,0.391255,0.101536,0.009646,0.001411,3,None,10,"{'max_depth': 3, 'max_features': None, 'n_esti...",0.860614,0.861237,0.865753,0.865597,0.860903,0.862821,0.002339,2
4,1.270551,0.080567,0.021771,0.001351,3,None,50,"{'max_depth': 3, 'max_features': None, 'n_esti...",0.860146,0.863728,0.863573,0.865286,0.863240,0.863195,0.001679,1


In [17]:
t1=icll_model.get_layer_search_time('L1')
t2=icll_model.get_layer_search_time('L2')

print(f"Time taken to perform hyperparameter tuning on Random Forest model: {np.round(t1,2)} seconds.")
print(f"Time taken for Logistic Regression model to train: {np.round(t2/60,2)} minutes.")

Time taken to perform hyperparameter tuning on Random Forest model: 17.03 seconds.
Time taken for Logistic Regression model to train: 0.57 minutes.


In [18]:
icll_model.get_computed_dist_threshold()

np.float64(1.7534645676717928)

In [19]:
cluster_type_info=icll_model.get_cluster_type_info()
print(cluster_type_info)

{'MAJORITY_CLUSTER_COUNT': 3124, 'MINORITY_CLUSTER_COUNT': 633, 'MIXED_CLUSTER_COUNT': 2250}


In [20]:
best_rf_model=icll_model.get_fitted_model_layer_object('L1')

feature_importance_list=best_rf_model.feature_importances_

importance_info_df=pd.DataFrame({'Feature Name': selected_columns,'Importance':feature_importance_list})
#importance_info_df.to_string()

In [21]:
fitted_lr_model=icll_model.get_fitted_model_layer_object('L2')

coeff_list=fitted_lr_model.coef_[0]

coef_df=pd.DataFrame({'Feature Name': selected_columns, 'Model Coefficients':coeff_list})
# coef_df.to_string()

In [22]:
coef_df

,Feature Name,Model Coefficients
0,age,0.007315
1,marital,-0.822288
2,default,-0.831480
3,housing,-1.197592
4,loan,-1.430569
5,day,-0.005099
6,campaign,-0.062564
7,pdays,0.004941
8,previous,0.271866
9,poutcome,1.196456


<h4></h4>